# CPU Workload Optimization (NumPy/Pandas)
VoltStream: Smart Grid Predictive Analytics Pipeline.

Note: Feel free to add Code/Markdown cells as you need.

# Part 1 : Working with RDDs (30%) <a class="anchor" name="part-1"></a>
## 1.1 Working with RDD
In this section, you will need to create RDDs from the given datasets, perform partitioning in these RDDs and use various RDD operations to answer the queries. 

1.1.1 Data Preparation and Loading <a class="anchor" name="1.1"></a>
1.	Write the code to create a SparkContext object using SparkSession. To create a SparkSession, you first need to build a SparkConf object that contains information about your application. Use Melbourne time as the session timezone. Give your application an appropriate name and run Spark locally with 4 cores on your machine.

In [1]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import json
import csv

# Spark configuration
conf = SparkConf()
conf.setAppName('Analysing Australian Property Market Data')  # App name
conf.setMaster('local[4]')                                    # Run locally with 4 cores
conf.set('spark.sql.session.timeZone', 'Australia/Melbourne') # Set time zone

# Create Spark session and context
spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext

# Spark is now ready for reading and processing data

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/14 07:54:25 WARN Utils: Your hostname, rhyme-server, resolves to a loopback address: 127.0.1.1; using 192.168.1.134 instead (on interface wlp0s20f3)
26/08/14 07:54:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/rhyme/repo/voltstream/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/14 07:54:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/14 07:54:27 WARN Utils: Service 'S

1.1.2 Load the CSV and JSON files into multiple RDDs. 

In [2]:
# Load main CSV file as an RDD
nsw_property_rdd = sc.textFile('../data/nsw_property_price.csv')

def parse_custom_json(file_content):
    """Parse JSON with a single top-level key containing the data list."""
    import json
    try:
        data = json.loads(file_content)
        return list(data.values())[0]  # Extract the inner list
    except (json.JSONDecodeError, IndexError):
        return []  # Return empty list on parse error

# Read JSON files and flatten their data
council_rdd = sc.wholeTextFiles('../data/council.json').flatMap(lambda x: parse_custom_json(x[1]))
property_purpose_rdd = sc.wholeTextFiles('../data/property_purpose.json').flatMap(lambda x: parse_custom_json(x[1]))
zoning_rdd = sc.wholeTextFiles('../data/zoning.json').flatMap(lambda x: parse_custom_json(x[1]))

1.1.3 For each RDD, remove the header rows and display the total count and the first 8 records.


In [3]:
# Process nsw_property_rdd
header_nsw = nsw_property_rdd.first()
nsw_property_rdd_no_header = nsw_property_rdd.filter(lambda row: row != header_nsw)
print(f"Total count of nsw_property_rdd: {nsw_property_rdd_no_header.count()}")
print("First 8 records of nsw_property_rdd:")
for record in nsw_property_rdd_no_header.take(8):
    print(record)

# Process council_rdd
print(f"\nTotal count of council_rdd: {council_rdd.count()}")
print("First 8 records of council_rdd:")
for record in council_rdd.take(8):
    print(record)

# Process property_purpose_rdd
print(f"\nTotal count of property_purpose_rdd: {property_purpose_rdd.count()}")
print("First 8 records of property_purpose_rdd:")
for record in property_purpose_rdd.take(8):
    print(record)

# Process zoning_rdd
print(f"\nTotal count of zoning_rdd: {zoning_rdd.count()}")
print("First 8 records of zoning_rdd:")
for record in zoning_rdd.take(8):
    print(record)

Total count of nsw_property_rdd: 4854814
First 8 records of nsw_property_rdd:
4270509,1400000.00,"8 C NYARI RD, KENTHURST","2156",house,,"",2.044,H,"2023-12-14","2024-02-14",V,"2/1229857",142,200,9922,53
4329326,1105000.00,"82 CAMARERO ST, BOX HILL","2765",house,,"",300.2,M,"2024-01-12","2024-02-09",R,"1119/1256791",143,200,7071,41
1864112,55000.00,"321 AUBURN ST, MOREE","2400",house,,"",847.3,M,"2023-09-15","2024-01-29",R,"17/36061",192,168,7071,40
1869899,680000.00,"207 GWYDIRFIELD RD, MOREE","2400",house,,SPRINGVALE,2.023,H,"2024-01-19","2024-02-09",R,"6/251911",193,168,7071,48
1867775,220000.00,"90 MERRIWA ST, BOGGABILLA","2409",house,,"",2023.0,M,"2023-12-08","2024-02-09",R,"1/1/758127",194,168,7071,52
2738374,690000.00,"10 PETOSTRUM PL, PORT MACQUARIE","2444",house,,"",672.8,M,"2023-12-14","2024-02-14",R,"94/815767",242,184,7071,40
1608665,661000.00,"71 MULYAN ST, COMO","2226",house,,"",561.7,M,"2013-03-23","2013-05-09","3","2/11301",26440,196,4301,2
638909,780208.00,"38 DUFFY AV

1.1.4 Drop records with invalid information: purpose_id or council_id is null, empty, or 0.

In [4]:
def parse_csv_line(line):
    return next(csv.reader([line]))

nsw_property_rdd_parsed = nsw_property_rdd_no_header.map(parse_csv_line)

def is_valid(record):
    try:
        purpose_id = record[16]
        council_id = record[15]
        return all([purpose_id, council_id, purpose_id != '0', council_id != '0'])
    except IndexError:
        return False

nsw_property_rdd_cleaned = nsw_property_rdd_parsed.filter(is_valid)
print(f"\nTotal count of nsw_property_rdd after cleaning: {nsw_property_rdd_cleaned.count()}")
print("First 8 records of cleaned nsw_property_rdd:")
for record in nsw_property_rdd_cleaned.take(8):
    print(record)

[Stage 9:================================================>        (16 + 3) / 19]


Total count of nsw_property_rdd after cleaning: 4836784
First 8 records of cleaned nsw_property_rdd:
['4270509', '1400000.00', '8 C NYARI RD, KENTHURST', '2156', 'house', '', '', '2.044', 'H', '2023-12-14', '2024-02-14', 'V', '2/1229857', '142', '200', '9922', '53']
['4329326', '1105000.00', '82 CAMARERO ST, BOX HILL', '2765', 'house', '', '', '300.2', 'M', '2024-01-12', '2024-02-09', 'R', '1119/1256791', '143', '200', '7071', '41']
['1864112', '55000.00', '321 AUBURN ST, MOREE', '2400', 'house', '', '', '847.3', 'M', '2023-09-15', '2024-01-29', 'R', '17/36061', '192', '168', '7071', '40']
['1869899', '680000.00', '207 GWYDIRFIELD RD, MOREE', '2400', 'house', '', 'SPRINGVALE', '2.023', 'H', '2024-01-19', '2024-02-09', 'R', '6/251911', '193', '168', '7071', '48']
['1867775', '220000.00', '90 MERRIWA ST, BOGGABILLA', '2409', 'house', '', '', '2023.0', 'M', '2023-12-08', '2024-02-09', 'R', '1/1/758127', '194', '168', '7071', '52']
['2738374', '690000.00', '10 PETOSTRUM PL, PORT MACQUARIE

### 1.2 Data Partitioning in RDD <a class="anchor" name="1.2"></a>
1.2.1 For each RDD, using Spark’s default partitioning, print out the total number of partitions and the number of records in each partition

In [5]:
# List of our RDDs and their names for easy iteration
rdds = {
    "nsw_property_rdd_cleaned": nsw_property_rdd_cleaned,
    "council_rdd": council_rdd,
    "property_purpose_rdd": property_purpose_rdd,
    "zoning_rdd": zoning_rdd
}

# Iterate through each RDD to get partitioning info
for name, rdd in rdds.items():
    num_partitions = rdd.getNumPartitions()
    # Use glom() to create an RDD of lists, where each list is a partition
    # Then map len to get the size of each partition
    partition_counts = rdd.glom().map(len).collect()
    
    print(f"--- {name} ---")
    print(f"Total number of partitions: {num_partitions}")
    # Enumerate to print the count for each partition number
    for i, count in enumerate(partition_counts):
        print(f"  Partition {i}: {count} records")
    print("-" * (len(name) + 8) + "\n")

[Stage 11:===============================================>        (16 + 3) / 19]

--- nsw_property_rdd_cleaned ---
Total number of partitions: 19
  Partition 0: 257607 records
  Partition 1: 256570 records
  Partition 2: 254864 records
  Partition 3: 255394 records
  Partition 4: 255727 records
  Partition 5: 258108 records
  Partition 6: 258790 records
  Partition 7: 257272 records
  Partition 8: 255254 records
  Partition 9: 254686 records
  Partition 10: 254416 records
  Partition 11: 253181 records
  Partition 12: 253382 records
  Partition 13: 255223 records
  Partition 14: 254220 records
  Partition 15: 257933 records
  Partition 16: 257370 records
  Partition 17: 255693 records
  Partition 18: 231094 records
--------------------------------

--- council_rdd ---
Total number of partitions: 1
  Partition 0: 220 records
-------------------

--- property_purpose_rdd ---
Total number of partitions: 1
  Partition 0: 865 records
----------------------------

--- zoning_rdd ---
Total number of partitions: 1
  Partition 0: 71 records
------------------



1.2.2 Answer the following questions:   
a) How many partitions do the above RDDs have?  
b) How is the data in these RDDs partitioned by default, when we do not explicitly specify any partitioning strategy? Can you explain why it is partitioned in this number?   
c) Data partitioning strategy.  

a) Based on the output from the code I ran, the RDDs have the following number of partitions:

nsw_property_rdd_cleaned: 19 partitions

council_rdd: 1 partition

property_purpose_rdd: 1 partition

zoning_rdd: 1 partition

b) When we don’t explicitly set a partitioning strategy, Spark chooses a default based on how the RDD was created.

For the nsw_property_rdd_cleaned, I created it by loading a large CSV file from disk using sc.textFile(). In this case, Spark automatically partitions the data based on the file size. Since I'm running in local mode with 4 cores, Spark divided the large file into 19 smaller chunks (partitions) to process them in parallel.

For the other three RDDs (council_rdd, property_purpose_rdd, and zoning_rdd), they were created from small JSON files. Because the files themselves are very small, Spark determined that each one only needed a single partition.



c) The default partitioning isn't great for querying by Property Price. Right now, properties with similar prices are probably scattered across all 19 partitions, meaning Spark would have to search through every partition for most price-related queries.

A much better approach would be to repartition the main RDD using Range Partitioning on the purchase_price.

How it works: We could define specific price ranges (like $0-$500k, $500k-$1M, etc.) and create new partitions where each one only contains properties within that price range.

Why it's better: This would group all the relevant data together. So, if I wanted to find houses between $500k and $1M, Spark could just go directly to the correct partition and ignore the others. This would be much faster because it avoids a full data scan and reduces shuffling across the nodes.

On my hardware: Since my machine is set up to run Spark with 4 cores, I would probably repartition the data into 4 or maybe 8 partitions. This would allow all my cores to work in parallel on price-based queries without creating too much overhead from managing lots of tiny partitions.

1.2.3 Create a user-defined function (UDF) to transform the date strings from ISO format (YYYY-MM-DD) (e.g. 2025-01-01) to Australian format (DD/Mon/YYYY) (e.g. 01/Jan/2025), then call the UDF to transform two date columns (iso_contract_date and iso_settlement_date) to contract_date and settlement_date.

In [6]:
from datetime import datetime

def transform_date_format(record):
    """
    UDF to transform iso_contract_date and iso_settlement_date to Australian format.
    """
    try:
        # Date columns are at index 9 and 10
        contract_date_iso = record[9]
        settlement_date_iso = record[10]
        
        # Parse the ISO date string (e.g., "2023-12-14")
        contract_date_obj = datetime.strptime(contract_date_iso, '%Y-%m-%d')
        settlement_date_obj = datetime.strptime(settlement_date_iso, '%Y-%m-%d')
        
        # Format into the desired Australian format (e.g., "14/Dec/2023")
        contract_date_aus = contract_date_obj.strftime('%d/%b/%Y')
        settlement_date_aus = settlement_date_obj.strftime('%d/%b/%Y')
        
        # Create a new list with the transformed dates
        # Note: We are replacing the old dates with the new ones.
        # Alternatively, you could append them as new columns.
        new_record = list(record) # Make a mutable copy
        new_record[9] = contract_date_aus
        new_record[10] = settlement_date_aus
        
        return tuple(new_record) # RDDs work best with immutable tuples
        
    except (ValueError, IndexError):
        # If date parsing fails or index is out of bounds, return the original record
        return record

# Apply the transformation function to our cleaned RDD
nsw_property_rdd_dates_transformed = nsw_property_rdd_cleaned.map(transform_date_format)

# Display the first 5 records to verify the transformation
print("First 5 records with transformed dates:")
for record in nsw_property_rdd_dates_transformed.take(5):
    print(record)

First 5 records with transformed dates:
('4270509', '1400000.00', '8 C NYARI RD, KENTHURST', '2156', 'house', '', '', '2.044', 'H', '14/Dec/2023', '14/Feb/2024', 'V', '2/1229857', '142', '200', '9922', '53')
('4329326', '1105000.00', '82 CAMARERO ST, BOX HILL', '2765', 'house', '', '', '300.2', 'M', '12/Jan/2024', '09/Feb/2024', 'R', '1119/1256791', '143', '200', '7071', '41')
('1864112', '55000.00', '321 AUBURN ST, MOREE', '2400', 'house', '', '', '847.3', 'M', '15/Sep/2023', '29/Jan/2024', 'R', '17/36061', '192', '168', '7071', '40')
('1869899', '680000.00', '207 GWYDIRFIELD RD, MOREE', '2400', 'house', '', 'SPRINGVALE', '2.023', 'H', '19/Jan/2024', '09/Feb/2024', 'R', '6/251911', '193', '168', '7071', '48')
('1867775', '220000.00', '90 MERRIWA ST, BOGGABILLA', '2409', 'house', '', '', '2023.0', 'M', '08/Dec/2023', '09/Feb/2024', 'R', '1/1/758127', '194', '168', '7071', '52')


### 1.3 Query/Analysis <a class="anchor" name="1.3"></a>
For this part, write relevant RDD operations to answer the following queries.

1.3.1 Extract the Month (Jan-Dec) information and print the total number of sales by contract date for each Month. (5%)

In [7]:
# 1.3.1 Extract Month and count sales

def safe_month_extractor(record):
    """
    Safely extracts the month from a record.
    Returns (Month, 1) on success, or None on failure.
    """
    try:
        # Check if the record is a list/tuple and has enough elements
        if isinstance(record, (list, tuple)) and len(record) > 9:
            # Attempt to split the date string. This will fail if the format is not DD/Mon/YYYY
            month = record[9].split('/')[1]
            return (month, 1)
        else:
            return None
    except IndexError:
        # This catches cases where the split does not produce at least 2 elements
        return None

# --- FIX: Use the safe extractor function and filter out failures ---
monthly_sales_rdd = nsw_property_rdd_dates_transformed.map(safe_month_extractor) \
                                                      .filter(lambda x: x is not None)

# The rest of the code remains the same
sales_by_month = monthly_sales_rdd.reduceByKey(lambda a, b: a + b)
sorted_sales = sales_by_month.sortBy(lambda x: x[1], ascending=False)

print("Total number of sales by contract month:")
for month, count in sorted_sales.collect():
    print(f"  {month}: {count} sales")

Total number of sales by contract month:
  Mar: 461519 sales
  May: 450172 sales
  Nov: 447462 sales
  Oct: 433103 sales
  Sep: 423994 sales
  Aug: 414212 sales
  Jun: 408592 sales
  Jul: 405107 sales
  Dec: 391570 sales
  Feb: 386002 sales
  Apr: 382872 sales
  Jan: 231686 sales


1.3.2 Which 5 councils have the largest number of houses? Show their name and the total number of houses. (Note: Each house may appear multiple times if there are more than one sales, you should only count them once.) (5%)

In [8]:
# 1.3.2 Find the 5 councils with the largest number of houses

# Step 1: Prepare the property data RDD with a robust filter
property_council_rdd = nsw_property_rdd_cleaned \
    .filter(lambda record: isinstance(record, (list, tuple)) and len(record) > 14) \
    .map(lambda record: (record[0], record[14])) \
    .distinct()

# Step 2: Count the number of unique houses per council
council_house_counts = property_council_rdd.map(lambda record: (record[1], 1)).reduceByKey(lambda a, b: a + b)

# Step 3: Prepare the council data RDD for joining
council_lookup_rdd = council_rdd.map(lambda record: (str(record['council_id']), record['council_name']))

# Step 4: Join the two RDDs
joined_rdd = council_house_counts.map(lambda x: (str(x[0]), x[1])).join(council_lookup_rdd)

# Step 5: Format the result and get the top 5
formatted_rdd = joined_rdd.map(lambda record: (record[1][1], record[1][0]))
top_5_councils = formatted_rdd.takeOrdered(5, key=lambda x: -x[1])

# Step 6: Print the final result
print("\nTop 5 councils with the largest number of unique houses:")
for council, count in top_5_councils:
    print(f"  {council}: {count} houses")

[Stage 24:=====================================================>  (18 + 1) / 19]


Top 5 councils with the largest number of unique houses:
  BLACKTOWN: 91214 houses
  LAKE MACQUARIE: 59118 houses
  THE HILLS SHIRE: 55033 houses
  LIVERPOOL: 49054 houses
  PENRITH: 46841 houses


## Part 2. Working with DataFrames (45%) <a class="anchor" name="2-dataframes"></a>
In this section, you need to load the given datasets into PySpark DataFrames and use DataFrame functions to answer the queries.
### 2.1 Data Preparation and Loading

2.1.1. Load the CSV/JSON files into separate dataframes. When you create your dataframes, please refer to the metadata file and think about the appropriate data type for each column.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import pyspark.sql.functions as F
import json

# 2.1.1. Load the CSV/JSON files into separate dataframes.

# --- FIX for CSV: Load with options to handle formatting issues ---
# We let Spark infer the schema initially and add an escape character option.
property_df = spark.read.csv(
    '../data/nsw_property_price.csv',
    header=True,
    inferSchema=True, # Let Spark figure out the types first
    escape='"' # Helps handle quotes within fields
)

# --- FIX for JSON: Manually parse the custom structure ---
def load_custom_json_to_df(file_path, spark_session):
    """Reads a custom-formatted JSON file and converts it to a DataFrame."""
    # Read the whole file as a single text record
    json_rdd = spark_session.sparkContext.wholeTextFiles(file_path).map(lambda x: x[1])
    # Parse the JSON string to get the list of records (which is the first value in the dict)
    data_list = json.loads(json_rdd.first()).values()
    # Parallelize the list of records into an RDD
    data_rdd = spark_session.sparkContext.parallelize(list(data_list)[0])
    # Convert the RDD of dictionaries to a DataFrame
    return spark_session.createDataFrame(data_rdd)

# Load each JSON file using our custom function
council_df = load_custom_json_to_df('../data/council.json', spark)
purpose_df = load_custom_json_to_df('../data/property_purpose.json', spark)
zoning_df = load_custom_json_to_df('../data/zoning.json', spark)


2.1.2 Display the schema of the dataframes.

In [10]:
print("--- Property DataFrame Schema ---")
property_df.printSchema()
print("\n--- Council DataFrame Schema ---")
council_df.printSchema()
print("\n--- Purpose DataFrame Schema ---")
purpose_df.printSchema()
print("\n--- Zoning DataFrame Schema ---")
zoning_df.printSchema()

--- Property DataFrame Schema ---
root
 |-- property_id: integer (nullable = true)
 |-- purchase_price: double (nullable = true)
 |-- address: string (nullable = true)
 |-- post_code: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- strata_lot_number: integer (nullable = true)
 |-- property_name: string (nullable = true)
 |-- area: double (nullable = true)
 |-- area_type: string (nullable = true)
 |-- iso_contract_date: date (nullable = true)
 |-- iso_settlement_date: date (nullable = true)
 |-- nature_of_property: string (nullable = true)
 |-- legal_description: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- council_id: integer (nullable = true)
 |-- purpose_id: integer (nullable = true)
 |-- zone_id: integer (nullable = true)


--- Council DataFrame Schema ---
root
 |-- council_id: long (nullable = true)
 |-- council_name: string (nullable = true)


--- Purpose DataFrame Schema ---
root
 |-- primary_purpose: string (nullable = true)
 |-- p

When the dataset is large, do you need all columns? How to optimize memory usage? Do you need a customized data partitioning strategy? (Note: Think about those questions but you don’t need to answer these questions.)

In [11]:
# Unify the area column to sqm
# 1 Hectare = 10000 sqm
# 1 Acre = 4000 sqm
# 1 M = 1 sqm

property_df_area_unified = property_df.withColumn(
    "area_sqm",
    F.when(F.col("area_type") == 'H', F.col("area") * 10000)
     .when(F.col("area_type") == 'A', F.col("area") * 4000)
     .otherwise(F.col("area"))
)

# Display the first 5 results to verify the new column
print("\nDataFrame with unified 'area_sqm' column:")
property_df_area_unified.select("area", "area_type", "area_sqm").show(5)


DataFrame with unified 'area_sqm' column:
+------+---------+--------+
|  area|area_type|area_sqm|
+------+---------+--------+
| 2.044|        H| 20440.0|
| 300.2|        M|   300.2|
| 847.3|        M|   847.3|
| 2.023|        H| 20230.0|
|2023.0|        M|  2023.0|
+------+---------+--------+
only showing top 5 rows


2.2.2. <pre>The top five property types are: Residence, Vacant Land, Commercial, Farm and Industrial.
However, for historical reason, they may have different strings in the database. Please update the primary_purpose with the following rules:
a)	Any purpose that has “HOME”, “HOUSE”, “UNIT” is classified as “Residence”;
b)	“Warehouse”, “Factory”,  “INDUST” should be changed to “Industrial”;
c)	Anything that contains “FARM”(i.e. FARMING), should be changed to “FARM”;
d)	“Vacant”, “Land” should be “Vacant Land”;
e)	Anything that has “COMM”, “Retail”, “Shop” or “Office” are “Cmmercial”.
f)	All remaining properties, including null and empty purposes, are classified as “Others”.
Show the count of each type in a table.
(note: Some properties are multi-purpose, e.g. “House & Farm”, it’s fine to count them multiple times.)
</pre>

In [12]:
# Step 1: Join and prepare the DataFrame, converting the purpose to uppercase for case-insensitive matching.
full_property_df = property_df_area_unified.join(purpose_df, "purpose_id", "left") \
                                           .withColumn("upper_purpose", F.upper(F.col("primary_purpose"))) \
                                           .fillna({"upper_purpose": ""})

# --- Step 2: Create a DataFrame for each category using more specific rules ---

# a) Residence: Add a condition to exclude industrial/commercial units from this category.
residence_df = full_property_df.filter(
    (F.col("upper_purpose").contains("HOME")) |
    (F.col("upper_purpose").contains("HOUSE")) |
    (F.col("upper_purpose").contains("RESIDENCE")) |
    (
        F.col("upper_purpose").contains("UNIT") &
        ~F.col("upper_purpose").contains("INDUST") &
        ~F.col("upper_purpose").contains("COMM") &
        ~F.col("upper_purpose").contains("FACTORY") &
        ~F.col("upper_purpose").contains("WAREHOUSE")
    )
).withColumn("category", F.lit("Residence"))

# b) Industrial
industrial_df = full_property_df.filter(
    (F.col("upper_purpose").contains("WAREHOUSE")) |
    (F.col("upper_purpose").contains("FACTORY")) |
    (F.col("upper_purpose").contains("INDUST"))
).withColumn("category", F.lit("Industrial"))

# c) Farm
farm_df = full_property_df.filter(
    F.col("upper_purpose").contains("FARM")
).withColumn("category", F.lit("Farm"))

# d) Vacant Land: Make the rule much stricter to avoid grabbing other land types.
vacant_land_df = full_property_df.filter(
    (F.col("upper_purpose").contains("VACANT LAND")) |
    (F.col("upper_purpose") == "LAND") |
    (F.col("upper_purpose") == "VACANT")
).withColumn("category", F.lit("Vacant Land"))

# e) Commercial
commercial_df = full_property_df.filter(
    (F.col("upper_purpose").contains("COMM")) |
    (F.col("upper_purpose").contains("RETAIL")) |
    (F.col("upper_purpose").contains("SHOP")) |
    (F.col("upper_purpose").contains("OFFICE"))
).withColumn("category", F.lit("Commercial"))

# --- Step 3: Union all categorized properties to handle multi-counting ---
all_categorized_df = residence_df.select("property_id", "category") \
    .unionAll(industrial_df.select("property_id", "category")) \
    .unionAll(farm_df.select("property_id", "category")) \
    .unionAll(vacant_land_df.select("property_id", "category")) \
    .unionAll(commercial_df.select("property_id", "category"))

# --- Step 4: Calculate counts for the main categories ---
main_category_counts = all_categorized_df.groupBy("category").count()

# --- Step 5: Identify and count the 'Others' ---
# This part remains the same: find all properties that were not matched by any of the above filters.
# This correctly includes null, empty, and unclassifiable purposes.
categorized_ids_df = all_categorized_df.select("property_id").distinct()
others_df = full_property_df.join(categorized_ids_df, "property_id", "left_anti")
others_count_df = others_df.select(F.lit("Others").alias("category")) \
                           .groupBy("category") \
                           .count()

# --- Step 6: Combine all counts and display the result ---
final_counts_df = main_category_counts.unionAll(others_count_df)

print("\\n--- Count of Each Property Type (Improved Logic) ---")
final_counts_df.orderBy(F.desc("count")).show()

\n--- Count of Each Property Type (Improved Logic) ---


[Stage 64:===========================================>              (3 + 1) / 4]

+-----------+-------+
|   category|  count|
+-----------+-------+
|  Residence|3903494|
|Vacant Land| 551077|
| Commercial| 136905|
|       Farm|  73952|
|     Others|  54952|
| Industrial|  37070|
+-----------+-------+



2.2.3 Find the top 20 properties that make the largest value gain, show their address, suburb, and value increased. To calculate the value gain, the property must have been sold multiple times, “value increase” can be calculated with the last sold price – first sold price, regardless the transactions in between. Print all 20 records.

In [13]:
# Step 1: Find the first and last sale date for each property
# We group by property_id and find the minimum and maximum contract dates.
date_extremes_df = property_df.groupBy("property_id") \
    .agg(
        F.min("iso_contract_date").alias("first_sale_date"),
        F.max("iso_contract_date").alias("last_sale_date")
    ) \
    .filter(F.col("first_sale_date") < F.col("last_sale_date")) # Ensure the property was sold at least twice

# Step 2: Join back to the original DataFrame to get the FIRST sale price
# We join on property_id AND the first_sale_date to find the row with the first sale.
first_sale_df = date_extremes_df.join(
    property_df,
    [
        date_extremes_df.property_id == property_df.property_id,
        date_extremes_df.first_sale_date == property_df.iso_contract_date
    ]
).select(
    date_extremes_df.property_id,
    "first_sale_date",
    "last_sale_date",
    F.col("purchase_price").alias("first_price"),
    "address"
)

# Step 3: Join back AGAIN to get the LAST sale price
# We join the result from Step 2 on property_id AND the last_sale_date.
last_sale_df = first_sale_df.join(
    property_df,
    [
        first_sale_df.property_id == property_df.property_id,
        first_sale_df.last_sale_date == property_df.iso_contract_date
    ]
).select(
    first_sale_df.property_id,
    first_sale_df.address,
    "first_price",
    F.col("purchase_price").alias("last_price")
)

# Step 4: Calculate the value increase and find the top 20
value_gain_df = last_sale_df.withColumn("value_increase", F.col("last_price") - F.col("first_price"))

top_20_properties = value_gain_df.orderBy(F.desc("value_increase")) \
                                 .limit(20)

# Step 5: Display the results
print("\n--- Top 20 Properties with Largest Value Gain ---")
top_20_properties.select("address", "value_increase").show(20, truncate=False)

26/08/14 07:56:29 WARN Column: Constructing trivially true equals predicate, 'property_id == property_id'. Perhaps you need to use aliases.
26/08/14 07:56:30 WARN Column: Constructing trivially true equals predicate, 'property_id == property_id'. Perhaps you need to use aliases.



--- Top 20 Properties with Largest Value Gain ---


[Stage 87:>                                                         (0 + 4) / 5]

+--------------------------------+--------------+
|address                         |value_increase|
+--------------------------------+--------------+
|8 ACACIA CCT, WARRIEWOOD        |8.7489E8      |
|38 BARRENJOEY RD, MONA VALE     |5.43102135E8  |
|1 FORBES RD, PARKES             |5.43102135E8  |
|358 ANZAC PDE, KINGSFORD        |5.43102135E8  |
|86 VICTORIA RD, ROZELLE         |5.43102135E8  |
|322 CANTERBURY RD, CANTERBURY   |5.4301596E8   |
|322 CANTERBURY RD, CANTERBURY   |5.4301596E8   |
|1234 PRINCES HWY, ENGADINE      |5.4301596E8   |
|1234 PRINCES HWY, ENGADINE      |5.4301596E8   |
|169 WILLOUGHBY RD, NAREMBURN    |5.4301596E8   |
|169 WILLOUGHBY RD, NAREMBURN    |5.4301596E8   |
|327 PRINCES HWY, ST PETERS      |5.4301596E8   |
|100 PACIFIC HWY, TUGGERAH       |5.42898135E8  |
|3986 PACIFIC HWY, GULMARRAD     |5.42714135E8  |
| RIVER ST, MACLEAN              |5.42694135E8  |
|136 PACIFIC HWY N, COFFS HARBOUR|5.42644135E8  |
|255 STEWART ST, BATHURST        |5.42633135E8  |


2.2.4 For each season, plot the median house price trend over the years. Seasons in Australia are defined as: (Spring: Sep-Nov, Summer: Dec-Feb, Autumn: Mar-May, Winter: Jun-Aug). 

In [14]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd

# Step 1: Add year and month columns, and filter for houses
property_df_with_dates = property_df.withColumn("year", F.year("iso_contract_date")) \
                                    .withColumn("month", F.month("iso_contract_date")) \
                                    .filter(F.col("property_type") == "house")

# Step 2: Define the seasons
seasonal_df = property_df_with_dates.withColumn(
    "season",
    F.when((F.col("month") >= 9) & (F.col("month") <= 11), "Spring")
     .when((F.col("month") == 12) | (F.col("month") <= 2), "Summer")
     .when((F.col("month") >= 3) & (F.col("month") <= 5), "Autumn")
     .when((F.col("month") >= 6) & (F.col("month") <= 8), "Winter")
     .otherwise("Unknown")
)

# Step 3: Filter for a reasonable date range and remove 'Unknown' seasons
cleaned_seasonal_df = seasonal_df.filter((F.col("year") >= 1880) & (F.col("season") != "Unknown"))

# Step 4: Group by year and season to calculate the median purchase price
median_price_df = cleaned_seasonal_df.groupBy("year", "season") \
    .agg(F.expr("percentile_approx(purchase_price, 0.5)").alias("median_price")) \
    .orderBy("year", "season")

# Step 5: Convert to Pandas DataFrame for plotting
pandas_df = median_price_df.toPandas()


# Step 6: Create a polished, professional plot

# Apply a clean and professional theme
sns.set_theme(style="whitegrid")

# Create the faceted plot
g = sns.relplot(
    data=pandas_df, 
    x="year", 
    y="median_price", 
    col="season",
    kind="line",
    col_wrap=2,
    marker="o",
    height=4,
    aspect=1.5
)

# 1. Format the Y-axis to show dollars in millions ($XM)
def millions_formatter(x, pos):
    # **FIX**: Removed the comma from 1,000,000
    return f'${x/1000000:.1f}M'

for ax in g.axes.flat:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(millions_formatter))
    # 2. Ensure X-axis ticks are integers
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    # Optional: Rotate for better spacing if needed
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')


# 3. Add a clear title and customize labels
g.fig.suptitle('Median House Price Trend by Season (2000-Present)', y=1.03, fontsize=16, weight='bold')
g.set_axis_labels("Year", "Median Purchase Price")
g.set_titles("Season: {col_name}", weight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97]) # Adjust layout to make space for suptitle
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

2.2.5 (Open Question) Explore the dataset freely and plot one diagram of your choice. Which columns (at least 2) are highly correlated to the sales price? Discuss the steps of your exploration and the results. (No word limit, please keep concise.) 

In [ ]:
from pyspark.sql import functions as F, types as T

# 1) Load raw CSV
df = (spark.read.csv('../data/nsw_property_price.csv', header=True, inferSchema=True)
      .withColumn("iso_contract_date", F.to_date("iso_contract_date"))
      .withColumn("purchase_price", F.col("purchase_price").cast("double"))
)

# 2) Compute area_sqm from (area, area_type)
df = df.withColumn(
    "area_sqm",
    F.when(F.col("area_type").isin("M", "SQM"), F.col("area").cast("double"))
     .when(F.col("area_type") == "H", F.col("area").cast("double") * F.lit(10000.0))  # hectares → m²
     .otherwise(None)
)

# 3) Derive a coarse property_category (tweak mappings as needed)
df = df.withColumn(
    "property_category_str",
    F.when(F.lower("property_type").rlike("house|apartment|unit|residen"), "Residence")
     .when(F.lower("property_type").rlike("vacant|land|lot"), "Vacant Land")
     .when(F.lower("property_type").rlike("industrial|warehouse|factory"), "Industrial")
     .when(F.lower("property_type").rlike("commercial|office|retail"), "Commercial")
     .when(F.lower("property_type").rlike("farm|rural|agri|acreage"), "Farm")
     .otherwise(None)
)

# 4) Convert to ARRAY type expected by your explode()
final_classified_df = df.withColumn("property_category", F.array("property_category_str"))

# (Optional) sanity check
final_classified_df.select("iso_contract_date","purchase_price","area_sqm","property_type","property_category").show(5, truncate=False)


In [ ]:
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.ticker import FuncFormatter

def analyze_value_trends(df):
    # Ensure types
    df = (df
          .withColumn("iso_contract_date", F.to_date("iso_contract_date"))
          .withColumn("purchase_price", F.col("purchase_price").cast("double"))
          .withColumn("area_sqm", F.col("area_sqm").cast("double")))

    # One row per category
    property_types_df = df.withColumn("property_category", F.explode("property_category"))

    # Keep main categories
    value_df = property_types_df.filter(
        F.col("property_category").isin(["Residence", "Vacant Land", "Industrial", "Commercial", "Farm"])
    )

    # Metrics + cleaning
    analysis_df = (value_df
                   .withColumn("year", F.year("iso_contract_date"))
                   .withColumn("price_per_sqm", F.col("purchase_price") / F.col("area_sqm")))

    cleaned_analysis_df = (analysis_df.filter(
        (F.col("year") >= 2000) & (F.col("year") <= 2024) &
        (F.col("price_per_sqm") > 10) & (F.col("price_per_sqm") < 50000) &
        (F.col("purchase_price") > 1000) & (F.col("area_sqm") > 0)
    ).na.drop(subset=["year", "price_per_sqm"]))

    # Annual median
    median_value_trends = (cleaned_analysis_df
        .groupBy("year", "property_category")
        .agg(F.expr("percentile_approx(price_per_sqm, 0.5)").alias("median_price_per_sqm"))
        .orderBy("year"))

    # Plot
    trends_pd = median_value_trends.toPandas()
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 8))
    sns.lineplot(data=trends_pd, x="year", y="median_price_per_sqm",
                 hue="property_category", marker='o', linewidth=2.5)
    plt.title('Market Value Growth by Property Category (2000-2024)', fontsize=16, weight='bold')
    plt.xlabel('Year'); plt.ylabel('Median Price per Square Meter ($)')
    plt.legend(title='Property Category')

    def currency_formatter(x, pos): return f'${x:,.0f}'
    plt.gca().yaxis.set_major_formatter(FuncFormatter(currency_formatter))
    plt.show()

# Use it with your actual DataFrame name:
analyze_value_trends(final_classified_df)


Write your dicsussion here.

### Part 3 RDDs vs DataFrame vs Spark SQL (25%) <a class="anchor" name="part-3"></a>
Implement the following complex queries using RDD, DataFrame in SparkSQL separately(choose two). Log the time taken for each query in each approach using the “%%time” built-in magic command in Jupyter Notebook and discuss the performance difference between these 2 approaches of your choice.
(notes: You can write a multi-step query or a single complex query, the choice is yours. You can reuse the data frame in Part 2.)

### a)	Implement the above query using two approaches of your choice separately and print the results. (Note: Outputs from both approaches of your choice are required, and the results should be the same.). 

#### 3.1. Implementation 1

In [ ]:
%%time
from pyspark.sql.functions import col, year, datediff, when, lit, expr

# --- Step 1: Filter the data (Corrected) ---

# FIX: Hardcode the last two years based on data inspection to avoid errors from bad dates.
# The data clearly shows the most recent full years of data are 2023 and 2024.
last_two_years = [2023, 2024]

# Apply the initial filters for the last 2 years, houses, and price < $2M
filtered_df = property_df.filter(
    (year("iso_contract_date").isin(last_two_years)) &
    (col("property_type") == "house") &
    (col("purchase_price") < 2000000)
)

# --- Step 2: Calculate settlement days and create buckets (No change needed here) ---

settlement_df = filtered_df.withColumn("settlement_days", datediff(col("iso_settlement_date"), col("iso_contract_date")))

bucketed_df = settlement_df.withColumn(
    "settlement_bucket",
    when(col("settlement_days") <= 15, "0-15 days")
    .when(col("settlement_days") <= 30, "16-30 days")
    .when(col("settlement_days") <= 45, "31-45 days")
    .when(col("settlement_days") <= 60, "46-60 days")
    .when(col("settlement_days") <= 90, "61-90 days")
    .otherwise("91+ days")
).withColumn(
    "price_bucket",
    when(col("purchase_price") < 500000, "0-$500K")
    .when(col("purchase_price") < 1000000, "$500K-$1M")
    .when(col("purchase_price") < 1500000, "$1M-$1.5M")
    .otherwise("$1.5M-$2M")
)

# --- Step 3: Group and Count (No change needed here) ---

transaction_counts_df = bucketed_df.filter(col("settlement_bucket") != "91+ days") \
                                   .groupBy(year("iso_contract_date").alias("year"), "price_bucket", "settlement_bucket") \
                                   .count()

# --- Step 4: Create a complete grid and join (No change needed here) ---

years = spark.createDataFrame([(y,) for y in last_two_years], ["year"])
price_buckets = spark.createDataFrame([("0-$500K",), ("$500K-$1M",), ("$1M-$1.5M",), ("$1.5M-$2M",)], ["price_bucket"])
settlement_buckets = spark.createDataFrame(
    [("0-15 days",), ("16-30 days",), ("31-45 days",), ("46-60 days",), ("61-90 days",)],
    ["settlement_bucket"]
)

complete_grid_df = years.crossJoin(price_buckets).crossJoin(settlement_buckets)

# Join the transaction counts with the complete grid
final_df_api = complete_grid_df.join(
    transaction_counts_df,
    ["year", "price_bucket", "settlement_bucket"],
    "left"
).na.fill(0)

# --- Step 5: Add Order Columns to the Final DataFrame and Sort (Corrected) ---

final_ordered_df = final_df_api.withColumn(
    "price_order",
    when(col("price_bucket") == "0-$500K", 1)
    .when(col("price_bucket") == "$500K-$1M", 2)
    .when(col("price_bucket") == "$1M-$1.5M", 3)
    .otherwise(4)
).withColumn(
    "settlement_order",
    when(col("settlement_bucket") == "0-15 days", 1)
    .when(col("settlement_bucket") == "16-30 days", 2)
    .when(col("settlement_bucket") == "31-45 days", 3)
    .when(col("settlement_bucket") == "46-60 days", 4)
    .otherwise(5)
)

# Sort by the new order columns and drop them before showing
final_ordered_df.orderBy("year", "price_order", "settlement_order") \
                .drop("price_order", "settlement_order") \
                .show(40)

#### 3.2. Implementation 2

In [ ]:
%%time
# --- Step 1: Register the DataFrame as a Temporary View ---
# This allows us to query the DataFrame using SQL
property_df.createOrReplaceTempView("properties")

# --- Step 2: Write and Execute the SQL Query ---
# FIX 1: Hardcode the years to 2023 and 2024 to avoid errors from bad data.
# FIX 2: Add a CASE statement to the ORDER BY clause to sort the buckets logically instead of alphabetically.

spark_sql_query = """
    WITH grid AS (
        SELECT
            year,
            price_bucket,
            settlement_bucket
        FROM
            (SELECT explode(array(2023, 2024)) as year)
            CROSS JOIN (SELECT explode(array('0-$500K', '$500K-$1M', '$1M-$1.5M', '$1.5M-$2M')) as price_bucket)
            CROSS JOIN (SELECT explode(array('0-15 days', '16-30 days', '31-45 days', '46-60 days', '61-90 days')) as settlement_bucket)
    ),
    transactions AS (
        SELECT
            YEAR(iso_contract_date) as year,
            CASE
                WHEN purchase_price < 500000 THEN '0-$500K'
                WHEN purchase_price < 1000000 THEN '$500K-$1M'
                WHEN purchase_price < 1500000 THEN '$1M-$1.5M'
                ELSE '$1.5M-$2M'
            END AS price_bucket,
            CASE
                WHEN DATEDIFF(iso_settlement_date, iso_contract_date) <= 15 THEN '0-15 days'
                WHEN DATEDIFF(iso_settlement_date, iso_contract_date) <= 30 THEN '16-30 days'
                WHEN DATEDIFF(iso_settlement_date, iso_contract_date) <= 45 THEN '31-45 days'
                WHEN DATEDIFF(iso_settlement_date, iso_contract_date) <= 60 THEN '46-60 days'
                WHEN DATEDIFF(iso_settlement_date, iso_contract_date) <= 90 THEN '61-90 days'
                ELSE '91+ days'
            END AS settlement_bucket
        FROM
            properties
        WHERE
            YEAR(iso_contract_date) IN (2023, 2024)
            AND property_type = 'house'
            AND purchase_price < 2000000
    ),
    counts AS (
        SELECT
            year,
            price_bucket,
            settlement_bucket,
            COUNT(*) as count
        FROM
            transactions
        WHERE
            settlement_bucket != '91+ days'
        GROUP BY
            year, price_bucket, settlement_bucket
    )
    SELECT
        g.year,
        g.price_bucket,
        g.settlement_bucket,
        COALESCE(c.count, 0) as count
    FROM
        grid g
    LEFT JOIN
        counts c ON g.year = c.year AND g.price_bucket = c.price_bucket AND g.settlement_bucket = c.settlement_bucket
    ORDER BY
        g.year,
        CASE g.price_bucket
            WHEN '0-$500K' THEN 1
            WHEN '$500K-$1M' THEN 2
            WHEN '$1M-$1.5M' THEN 3
            ELSE 4
        END,
        CASE g.settlement_bucket
            WHEN '0-15 days' THEN 1
            WHEN '16-30 days' THEN 2
            WHEN '31-45 days' THEN 3
            WHEN '46-60 days' THEN 4
            ELSE 5
        END
"""

final_df_sql = spark.sql(spark_sql_query)

# --- Step 3: Display the result ---
final_df_sql.show(40)

### b)	Which one is easier to implement, in your opinion? Log the time taken for each query, and observe the query execution time, among DataFrame and SparkSQL, which is faster and why? Please include proper references. (Maximum 500 words.) 

### Some ideas on the comparison

Armbrust, M., Huai, Y., Liang, C., Xin, R., & Zaharia, M. (2015). Deep Dive into Spark SQL’s Catalyst Optimizer. Retrieved September 30, 2017, from https://databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html

Damji, J. (2016). A Tale of Three Apache Spark APIs: RDDs, DataFrames, and Datasets. Retrieved September 28, 2017, from https://databricks.com/blog/2016/07/14/a-tale-of-three-apache-spark-apis-rdds-dataframes-and-datasets.html

Data Flair (2017a). Apache Spark RDD vs DataFrame vs DataSet. Retrieved September 28, 2017, from http://data-flair.training/blogs/apache-spark-rdd-vs-dataframe-vs-dataset

Prakash, C. (2016). Apache Spark: RDD vs Dataframe vs Dataset. Retrieved September 28, 2017, from http://why-not-learn-something.blogspot.com.au/2016/07/apache-spark-rdd-vs-dataframe-vs-dataset.html

Xin, R., & Rosen, J. (2015). Project Tungsten: Bringing Apache Spark Closer to Bare Metal. Retrieved September 30, 2017, from https://databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html